In [ ]:
# 1. requests

In [ ]:
# 获取cookie

import time
from selenium import webdriver
from selenium.webdriver.common.by import By

def GetCookie(url):
    
    driver = webdriver.Chrome()
    driver.maximize_window()  # 最大化窗口
    
    # 打开网页
    driver.get(url)
    
    time.sleep(50)
    driver.refresh()
    
    cookies = driver.get_cookies()

    # 输出 cookies
    #for cookie in cookies:
    #    print(cookie)
    # 将 cookies 转换为 header 格式 :  Apache=349490162003.9359.1738739237863
    cookie_string = '; '.join([f"{cookie['name']}={cookie['value']}" for cookie in cookies])
    print(cookie_string)
    return cookie_string
    
url = "https://weibo.com"
cookie_string = GetCookie(url)

WBPSESS=VThGGM3X7D1UVj2-M0jRrxvm4lhERR4YTI3s0CGZqBbbqJRiiEzvGQuLLjh0EYsUkPZAHlMPhHFh7q1DGoUifsdeUBlWdNfr4oIcVVe43sQmpP-pq6Njlw2fjlS57_w7WF_xO6iCjv3z6CKj9VSnIQ==; SCF=AsAquVvx-sqeXJ562xRbkPQfYq0sOuPAopVVkotWoq3S_qXvtFCESxFlATj98zJAo-kfmzJx4vMjXO9eYq2ohjI.; SUBP=0033WrSXqPxfM725Ws9jqgMF55529P9D9Whdy5ceXwTvYqa___S8JwmI5JpX5KzhUgL.FoM01Kn0S0MEeK52dJLoIXnLxKBLB.2L1-2LxK-L1h-LBo.LxKqL12zL1h.LxKqL1KqL1K5LxK-LBo5L1KBLxKqLBozLBK2LxKqL1-eL1h.LxKML12zL1KMt; ALF=02_1743271084; SUB=_2A25KxNf8DeRhGeFN4loS9ynOyjyIHXVpuFU0rDV8PUNbmtANLRSnkW9NQ7hLS5oxJWiBmGHdwWaBHYalNYTHCXpm; XSRF-TOKEN=l4z5eHVznMPQBH6c3oiMaHor


In [ ]:
# 爬取微博一页数据

import requests
from bs4 import BeautifulSoup as BS
def TestOnePage(keyword,page):
                url = f'https://s.weibo.com/weibo?q={keyword}&page={page}'
                headers = {
                    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
                    'accept-encoding': 'gzip, deflate, br',
                    'accept-language': 'zh-CN,zh;q=0.9',
                    'cache-control': 'max-age=0',
                    # 根据上面执行结果获得
                    'cookie': cookie_string,
                    'sec-ch-ua': '" Not A;Brand";v="99", "Chromium";v="98", "Google Chrome";v="98"',
                    'sec-ch-ua-mobile': '?0',
                    'sec-ch-ua-platform': '"Windows"',
                    'sec-fetch-dest': 'document',
                    'sec-fetch-mode': 'navigate',
                    'sec-fetch-site': 'none',
                    'sec-fetch-user': '?1',
                    'upgrade-insecure-requests': '1',
                    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36',
                }
                ss = requests.session()
                ss.keep_alive = False
                while 1:
                    try:
                        html0 = ss.get(url=url, headers=headers, timeout=(3, 3)).text
                        break
                    except:
                        print("无法正确获取页面")
                        pass

                t = BS(html0, 'lxml')
                try:
                    cards = t.find_all('div', class_="card-wrap")
                    for card in cards:
                        
                        name = card.find('a', class_="name").text
                        
                        # 查找发博时间
                        timeTxt = str(card.find('div', class_="from").text).replace('\n','')
                        time = timeTxt.split(' ')
                        time = list(filter(lambda x: x != '', time))
                        time = time[0] + ',' + time[1]
                        
                        # 查找正文内容
                        feed_list_content = card.find('p', attrs={'node-type': 'feed_list_content'})
                        if feed_list_content:
                            # 提取<p>内所有非 <a> 标签的文本内容
                            text_content = ""
                            for element in feed_list_content.children:
                                if isinstance(element, str):  # 如果是文本节点
                                    text_content += element.strip()
                                elif element.name != 'a':  # 跳过 <a> 标签
                                    text_content += element.get_text(strip=True)
                        info = []
                        info.append(keyword)
                        info.append(name)
                        info.append(time)  
                        info.append(text_content)  
                        print(info) 
                        
                except:
                    print("There is no info in this page")
                
                
keyword =  '玉渊潭'                        
TestOnePage(keyword,1)

In [ ]:
# 读写数据

import os
import xlrd
from xlutils.copy import copy
import xlwt


path = 'keyword.xlsx'
weibo_cols = ['关键词','发博用户','发博时间','博文内容']
redbook_cols = ['关键词','发博用户','博文内容','链接']
def WriteNewDataToSheet(path,value,data_cols):
        """
        插入数据到表格里面
        :param path : 具体的文件地址
        :param value : 具体的数据
        :param data_cols : 想要插入的数据列
        """
        
        if not os.path.exists(path):  
            new_workbook = xlwt.Workbook()
            new_worksheet = new_workbook.add_sheet("Sheet1")
            for col, col_name in enumerate(data_cols):
                new_worksheet.write(0, col, col_name)  
            new_workbook.save(path)
    
       
        workbook = xlrd.open_workbook(path, formatting_info=True)
        sheets = workbook.sheet_names()
        
        worksheet = workbook.sheet_by_name(sheets[0])  
        rows_old = worksheet.nrows 
        new_workbook = copy(workbook)
        new_worksheet = new_workbook.get_sheet(0)
        for i in range(1, rows_old):
            if value[1] == worksheet.cell_value(i, 1) and value[2] == worksheet.cell_value(i, 2): 
                return  
        for j in range(len(data_cols)):
            new_worksheet.write(rows_old, j, value[j]) 

        new_workbook.save(path) 
  

In [ ]:
# 爬取多页数据
def GetInfoInWeibo(keyword):
                
                infos = []
                # 获取前10页的信息
                for page in range(1,31):
                    url = f'https://s.weibo.com/weibo?q={keyword}&page={page}'
                    headers = {
                        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
                        'accept-encoding': 'gzip, deflate, br',
                        'accept-language': 'zh-CN,zh;q=0.9',
                        'cache-control': 'max-age=0',
                        # 这里需要输入自己登陆的时候的cookie
                        'cookie': cookie_string,
                        'sec-ch-ua': '" Not A;Brand";v="99", "Chromium";v="98", "Google Chrome";v="98"',
                        'sec-ch-ua-mobile': '?0',
                        'sec-ch-ua-platform': '"Windows"',
                        'sec-fetch-dest': 'document',
                        'sec-fetch-mode': 'navigate',
                        'sec-fetch-site': 'none',
                        'sec-fetch-user': '?1',
                        'upgrade-insecure-requests': '1',
                        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36',
                    }
                    ss = requests.session()
                    ss.keep_alive = False
                    while 1:
                        try:
                            html0 = ss.get(url=url, headers=headers, timeout=(3, 3)).text
                            break
                        except:
                            pass

                    t = BS(html0, 'lxml')
                    try:
                        pdd = t.find('div', class_="card card-no-result s-pt20b40")  # 如果找没有信息到直接跳出程序
                        if pdd != None:
                            break
                    except:
                        print("It's the last Page")
                        
                    cards = t.find_all('div', class_="card-wrap")
                    for card in cards:
                        try:
                            # 查找id
                            name = card.find('a', class_="name").text
                            
                            # 查找发博时间
                            timeTxt = str(card.find('div', class_="from").text).replace('\n','')#.replace(' ', '').replace('\n', '').replace('\xa0','')
                            time = timeTxt.split(' ')
                            time = list(filter(lambda x: x != '', time))
                            time = time[0] + ',' + time[1]
                            
                            feed_list_content = card.find('p', attrs={'node-type': 'feed_list_content'})
                            if feed_list_content:
                                # 提取<p>内所有非 <a> 标签的文本内容
                                text_content = ""
                                for element in feed_list_content.children:
                                    if isinstance(element, str):  # 如果是文本节点
                                        text_content += element.strip() # “ ”\t” "\n"
                                    elif element.name != 'a':  # 跳过 <a> 标签
                                        text_content += element.get_text(strip=True)
                            info = []
                            info.append(keyword)
                            info.append(name)
                            info.append(time)  
                            info.append(text_content)  
                            infos.append(info)
                        except:
                            pass
                print(infos)
                path = 'weibo_data1.xlsx'
                for info in infos:
                    WriteNewDataToSheet(path,info,weibo_cols)
park_list = ['玉渊潭','天坛','香山','陶然亭','朝阳公园','颐和园','北海公园']
for park in park_list:                        
    GetInfoInWeibo(park)

In [ ]:
# 如果要涉及时间
# https://s.weibo.com/weibo?q=颐和园&typeall=1&suball=1&timescope=custom%3A2025-02-01-0%3A2025-02-25-0&Refer=g
# 2025-02-01-0 -》 2025-02-25-0
import datetime
# 开始时间
start_date = '2022-12-01'
# 结束时间
end_date = '2023-02-28'

dates = []
dt = datetime.datetime.strptime(start_date, "%Y-%m-%d")
date = start_date[:]
while date <= end_date:
    dates.append(date)
    dt = dt + datetime.timedelta(1)
    date = dt.strftime("%Y-%m-%d")
  
for date in dates:
    for i in range(1, 51):
        url = f'https://s.weibo.com/weibo?q={keyword}&typeall=1&suball=1&timescope=custom%3A{date}-0%3A{date}-23&Refer=g&page={i}'

In [ ]:
# 2. 小红书

In [ ]:
# 关键词编码

import urllib.parse

def double_url_encode(text):
    # 第一次 URL 编码
    encoded_once = urllib.parse.quote(text, encoding='utf-8')
    # 第二次 URL 编码
    encoded_twice = urllib.parse.quote(encoded_once, encoding='utf-8')
    return encoded_twice

keyword = '玉渊潭'
keywordUrl = double_url_encode(keyword)
print(keywordUrl)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


keyword = '玉渊潭'
keywordUrl = double_url_encode(keyword)

driver = webdriver.Chrome()
url = f'https://www.xiaohongshu.com/search_result/?keyword={keywordUrl}'
# 打开网页
driver.get(url)

# 手动登陆
time.sleep(20)

# 获取页面的高度
last_height = driver.execute_script("return document.body.scrollHeight")
href_list = []
while True:
    # 滚动页面，垂直方向
     driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    
    # 等待页面加载
     time.sleep(5)
     try:
          # 查找所有 class="cover ld mask" 的 <a> 标签
          elements = WebDriverWait(driver, 10).until(
               EC.presence_of_element_located((By.CSS_SELECTOR, "a.cover.ld.mask"))
          )
          elements = driver.find_elements(By.CSS_SELECTOR, "a.cover.ld.mask")
          # 提取所有 href 链接
          href_list += [element.get_attribute("href") for element in elements]
          
     except:
          print("不能找到任何信息")
          pass

    # 获取新页面的高度
     new_height = driver.execute_script("return document.body.scrollHeight")
     
    # 判断是否滚动到页面底部
     if new_height == last_height:
          break
     last_height = new_height

# 输出结果
print(href_list)

In [ ]:
import random

infos = []
for url in href_list:
    headers = {
                        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
                        'accept-encoding': 'gzip, deflate, br',
                        'accept-language': 'zh-CN,zh;q=0.9',
                        'cache-control': 'max-age=0',
                        # 这里需要输入自己登陆的时候的cookie
                        'cookie': cookie_string,
                        'sec-ch-ua': '" Not A;Brand";v="99", "Chromium";v="98", "Google Chrome";v="98"',
                        'sec-ch-ua-mobile': '?0',
                        'sec-ch-ua-platform': '"Windows"',
                        'sec-fetch-dest': 'document',
                        'sec-fetch-mode': 'navigate',
                        'sec-fetch-site': 'none',
                        'sec-fetch-user': '?1',
                        'upgrade-insecure-requests': '1',
                        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36',
    }
    headers_list = [
        {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
            "Referer": "https://www.google.com/",
            "Accept-Language": "en-US,en;q=0.9",
            'cookie': cookie_string,
            "Connection": "keep-alive",
        },
        {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
            "Referer": "https://www.bing.com/",
            "Accept-Language": "en-GB,en;q=0.8",
            'cookie': cookie_string,
            "Connection": "keep-alive",
        },
        {
            "User-Agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 16_0 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile/15E148 Safari/604.1",
            "Referer": "https://m.facebook.com/",
            "Accept-Language": "en-US,en;q=0.7",
            'cookie': cookie_string,
            "Connection": "keep-alive",
        },
    ]

    # 随机选择一个 header
    headers = random.choice(headers_list)
    time.sleep(random.uniform(1, 5)) # 随机等待时间
    ss = requests.session()
    ss.keep_alive = False
    while 1:
        try:
            html0 = ss.get(url=url, headers=headers, timeout=(3, 3)).text
            break
        except:
            pass

    t = BS(html0, 'lxml')
    try:
        #detail-desc
        note = t.find('div', class_="detail-desc")  # 如果找没有信息到直接跳出程序
        #username
        username = t.find('span', class_="username")
        # note
        noteText = t.find('span', class_="note-text")
        info = []
        username_text = username.get_text(strip=True)  # 获取 username 的文本
        note_text_content = noteText.get_text(strip=True)  # 获取 note-text 的文本
        info.append(keyword)
        info.append(username_text)
        info.append(note_text_content)
        info.append(url)
        infos.append(info)
    except:
        print(f"内容没有正确显示:{url}")

path = 'redbook_data.xlsx'
for info in infos:
    WriteNewDataToSheet(path,info,redbook_cols) 

In [ ]:
# 获取cookie 

url = "https://www.xiaohongshu.com/"
driver = webdriver.Chrome()
driver.maximize_window()  # 最大化窗口
    
# 打开网页
driver.get(url)
    
time.sleep(30)
    
# 输入你的用户名和密码
#获取更新的cookie
driver.refresh()
cookies = driver.get_cookies()
driver.quit()

In [ ]:
# 时间转换

from datetime import datetime, timedelta
import re

def convert_relative_time(relative_time):
    """
    将 '3天前' / ‘昨天’这种相对时间转换为具体的日期时间
    """
    # 获取当前时间
    now = datetime.now()
    
    if("昨天" in relative_time):
        return now - timedelta(days=1)
    
    # 匹配时间单位
    match = re.match(r"(\d+)([天])前", relative_time)
    
    if match:
        num = int(match.group(1))  # 数字
        return now - timedelta(days=num)
    
    
    return None  # 无法解析的格式返回 None


time_str = "昨天"
converted_time = convert_relative_time(time_str)
print("转换后的时间:", converted_time.strftime("%Y-%m-%d"))

In [ ]:
keyword = "玉渊潭"
infos=[]
for url in href_list:
    driver = webdriver.Chrome()
    driver.maximize_window()  # 最大化窗口
    driver.get(url)
    for cookie in cookies:
        driver.add_cookie(cookie)
    try:
        username = driver.find_element(By.XPATH, "//*[@id='noteContainer']/div[4]/div[1]/div/div[1]/a[2]/span").text
        note_text = driver.find_element(By.XPATH, "//span[@class='note-text']/span").text
        time_text = driver.find_element(By.XPATH, "//*[@id='noteContainer']/div[4]/div[2]/div[1]/div[3]/span[1]").text
        if('天' in time_text):
            # 7 天前 北京
            time_stamps = time_text.split(' ') # ['7', '天前', '北京']
            if(len(time_stamps)  ==  3): #2-7天前
                time_text = time_stamps[0] + time_stamps[1]  # 7天前
            else: # 昨天
                time_text = time_stamps[0]
            converted_time = convert_relative_time(time_text) # date
            time_text = converted_time.strftime("%Y-%m-%d") # 2024-12-07
        info = []
        info.append(keyword)
        info.append(username)
        info.append(time_text)
        info.append(note_text)
        
        info.append(url)
        infos.append(info)
    except:
          print(f"内容没有正确显示:{url}")
          pass
print(infos)
path = 'redbook_data.xlsx'
for info in infos:
    WriteNewDataToSheet(path,info,redbook_cols)    